# 07-units (Medellin, PM2.5) -- dimensionally constrained search

Same idea as `bogota/07_units_equation_discovery.ipynb`: the pool is the existing same-day no-lag selection (so it is the free-form run's pool), every variable gets a unit, and PySR is restricted to dimensionally homogeneous expressions. Units: gases and PM as mass concentrations (`kg/m^3`, the documented assumption of the units audit, A4), `ventilation_station` = BLH x wind speed (`m^2/s`), `tke_proxy` = 0.5 (var u10 + var v10) (`m^2/s^2`), fire counts dimensionless. Three seeds.

Scoring (same protocol as the Bogota lead equations, notebooks A2/A4): leave-one-year-out, constants refitted by non-linear least squares on the training years, **both** pooled R2 and mean of per-year R2, on the rows the search used. The existing free-form front is scored on the same rows.

In [1]:
# %% Settings (edit here, then Run All)
CITY, TARGET, TAG = 'medellin', 'pm25', 'units'
MASTER = 'medellin_master_dataset.parquet'                         # file in Data/processed/colombia/<city>/
SELECTION_JSON = 'medellin_feature_selection.json'                    # its selected_no_lag is the pool (identical to the free-form run)
EXTRA_FEATURES = []                  # added to that pool (none added)
DROP_FEATURES = []                    # removed from it (split into raw factors)
SEEDS = [0, 1, 2]                           # symbolic search is stochastic: report all of them
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to the Bogota constrained runs
UNIT_MAP = {'hotspots_sector_E_lag3': '', 'no_lag3': 'kg/m^3', 'no2': 'kg/m^3', 'ventilation_station': 'm^2/s', 'tke_proxy': 'm^2/s^2'}
Y_UNITS = 'kg/m^3'
FREEFORM_FRONT = 'medellin_pysr_pareto_front.csv'                     # the existing free-form front, scored here on the same rows
MAX_COMPLEXITY_REPORTED = 25

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / CITY
df = pd.read_parquet(DATA_DIR / MASTER)
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
selection = json.load(open(DATA_DIR / SELECTION_JSON))
features = [f for f in selection["selected_no_lag"] if f not in DROP_FEATURES]
for f in EXTRA_FEATURES:
    if f not in features: features.append(f)
print("pool:", features)
data = df.dropna(subset=features + [TARGET]).reset_index(drop=True)
X = data[features].to_numpy(); y = data[TARGET].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)), "years:", sorted(data.year.unique()))

pool: ['hotspots_sector_E_lag3', 'no_lag3', 'no2', 'ventilation_station', 'tke_proxy']
(2845, 257) {'hotspots_sector_E_lag3': '', 'no_lag3': 'kg/m^3', 'no2': 'kg/m^3', 'ventilation_station': 'm^2/s', 'tke_proxy': 'm^2/s^2'} years: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[["complexity", "loss", "equation"]].copy(); f["seed"] = seed
    f.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.770e+04
Progress: 40 / 200 total iterations (20.208%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.054e+01  0.000e+00  y[m⁻³ kg] = 19.957[?]
3           4.767e+01  2.928e-02  y[m⁻³ kg] = no2[m⁻³ kg] * 0.54325[?]
4           4.300e+01  1.031e-01  y[m⁻³ kg] = sqrt(no2[m⁻³ kg] * 11.37[?])
5           4.286e+01  3.116e-03  y[m⁻³ kg] = (no2[m⁻³ kg] + 29.437[?]) * 0.30555[?]
7           4.286e+01  1.812e-05  y[m⁻³ kg] = ((no2[m⁻³ kg] + 58.88[?]) + no2[m⁻³ kg...
                                              ]) * 0.15311[?]
8           4.084e+01  4.825e-02  y[m⁻³ kg] = sqrt((no_lag3[m⁻³ kg] + (no2[m⁻³ kg] *...
                                               -5.3651[?])) * -2.5425[?])
10          4.002e+01  1.018e-02  y[m⁻³ kg] = ((sqrt(no_lag3[m⁻³ k

seed 0 done; best: sqrt(no2 / 0.08807528)

Expressions evaluated per second: 1.720e+04
Progress: 39 / 200 total iterations (19.750%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.054e+01  0.000e+00  y[m⁻³ kg] = 19.959[?]
3           4.767e+01  2.926e-02  y[m⁻³ kg] = no2[m⁻³ kg] * 0.54447[?]
4           4.300e+01  1.032e-01  y[m⁻³ kg] = sqrt(no2[m⁻³ kg] / 0.088099[?])
5           4.286e+01  3.065e-03  y[m⁻³ kg] = (no2[m⁻³ kg] / 3.3139[?]) + 9.2033[?]
7           4.261e+01  3.023e-03  y[m⁻³ kg] = (no2[m⁻³ kg] + (1.2488[?] / tke_proxy[...
                                              m² s⁻²])) * 0.43077[?]
8           4.230e+01  7.194e-03  y[m⁻³ kg] = sqrt(no2[m⁻³ kg] / sqrt(0.16303[?] / s...
                                              qrt(ventilation_station[m² s⁻¹])))
9       

/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


�═════════════════════════════════════════

Expressions evaluated per second: 1.830e+04
Progress: 83 / 200 total iterations (41.604%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.054e+01  0.000e+00  y[m⁻³ kg] = 19.959[?]
3           4.767e+01  2.928e-02  y[m⁻³ kg] = no2[m⁻³ kg] * 0.54351[?]
4           4.300e+01  1.031e-01  y[m⁻³ kg] = sqrt(no2[m⁻³ kg] / 0.088099[?])
5           4.286e+01  3.103e-03  y[m⁻³ kg] = (no2[m⁻³ kg] / 3.2574[?]) + 9.0284[?]
7           4.257e+01  3.401e-03  y[m⁻³ kg] = (0.44392[?] / tke_proxy[m² s⁻²]) + (no...
                                              2[m⁻³ kg] / 2.2165[?])
8           4.029e+01  5.522e-02  y[m⁻³ kg] = sqrt(no2[m⁻³ kg] / sqrt(sqrt(no_lag3[m...
                                              ⁻³ kg] * 2.5086e-06[?])))
10          3.99

/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.740e+04
Progress: 40 / 200 total iterations (20.458%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.054e+01  0.000e+00  y[m⁻³ kg] = 19.957[?]
3           4.767e+01  2.927e-02  y[m⁻³ kg] = no2[m⁻³ kg] / 1.8435[?]
4           4.300e+01  1.031e-01  y[m⁻³ kg] = sqrt(no2[m⁻³ kg] * 11.355[?])
5           4.287e+01  3.028e-03  y[m⁻³ kg] = (no2[m⁻³ kg] / 3.1721[?]) + 8.6669[?]
7           4.286e+01  4.178e-05  y[m⁻³ kg] = ((no2[m⁻³ kg] + -8.5045[?]) / 3.256[?]...
                                              ) + 11.549[?]
8           4.078e+01  4.974e-02  y[m⁻³ kg] = sqrt((no_lag3[m⁻³ kg] * -3.0204[?]) + ...
                                              (no2[m⁻³ kg] * 13.314[?]))
9           4.000e+01  1.946e-02  y[m⁻³ kg] = ((no_lag3[m⁻³ kg] / -7.7

In [4]:
FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

yy = data[TARGET].to_numpy(float); years = data["year"].to_numpy(); YRS = sorted(set(years))
def score(eq, feats):
    model, used, p0 = make_model(eq, feats)
    if not used:                                   # constant equation: leave-one-year-out mean
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in YRS]); yo = np.concatenate([yy[years == yr] for yr in YRS])
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in YRS]
        return dict(r2_insample=0.0, r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(data[f].to_numpy(float) for f in used)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in YRS:
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred)); ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in YRS] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan, r2_pooled=r2_score(yy, pred) if ok else np.nan,
                    r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity <= MAX_COMPLEXITY_REPORTED: rows.append(dict(seed=str(seed), complexity=int(r.complexity), equation=r.equation, **score(r.equation, features)))
ff = pd.read_csv(DATA_DIR / FREEFORM_FRONT).sort_values("complexity")
ff_feats = sorted({t for e in ff.equation for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", e) if t in data.columns})
for _, r in ff.iterrows():
    if r.complexity <= MAX_COMPLEXITY_REPORTED:
        try: rows.append(dict(seed="free-form", complexity=int(r.complexity), equation=r.equation, **score(r.equation, ff_feats)))
        except Exception as e: print("free-form eq skipped", r.complexity, e)
scored = pd.DataFrame(rows); scored.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print("n days", len(data)); print(scored[["seed", "complexity", "r2_pooled", "r2_mean_years", "vars"]].round(3).to_string(index=False))

n days 2845
     seed  complexity  r2_pooled  r2_mean_years                                                             vars
        0           1     -0.061         -0.444                                                                 
        0           3      0.010         -0.357                                                              no2
        0           4      0.113         -0.209                                                              no2
        0           5      0.114         -0.208                                                              no2
        0           8        NaN            NaN                                                      no_lag3,no2
        0           9      0.179         -0.083                                                      no_lag3,no2
        0          10      0.179         -0.081                                                      no_lag3,no2
        0          11      0.189         -0.083                                     